In [0]:
-- ============================================================================
-- mkt_dashboard_tickets_monthly
-- Grain: report_month x population_type (CC / Scene / Total_Biz) -- 3 rows/month
-- Parameterized on :report_month (first-of-month DATE, e.g. '2026-07-01')
-- ============================================================================

WITH params AS (
    SELECT DATE('2026-08-01') AS report_month  -- TEMP: hardcoded for testing, revert to DATE(:report_month)
),

month_bounds AS (
    SELECT
        report_month                                   AS month_start,
        LAST_DAY(report_month)                          AS month_end,
        ADD_MONTHS(report_month, -1)                    AS prior_month_start,
        LAST_DAY(ADD_MONTHS(report_month, -1))          AS prior_month_end
    FROM params
),

-- ----------------------------------------------------------------------------
-- Ticket-line grain: EXH only, joined to session for format classification.
-- Date dimension is joined on TransactionDateId (not VisitDateId) -- the
-- whole table now reports on transaction month, not visit month.
-- No CDE_ID filter here -- Total_Biz needs anonymous/unidentified ticket
-- buyers included. CC/Scene branches apply CDE_ID IS NOT NULL themselves.
-- ----------------------------------------------------------------------------
ticket_events AS (
    SELECT
        t.CDE_ID                            AS cde_id,
        d.datefull                          AS datefull,
        t.CineClubFLAG                      AS cineclub_flag,
        t.SCENEFLAG                         AS scene_flag,
        tk.paid_attendance_inflag           AS paid_flag,
        t.CineClub_Discount_Type_ID         AS discount_type_id,
        t.Quantity                          AS quantity,
        s.Auditorium_Experience_ID          AS auditorium_experience_id,
        s.Performance_Experience_ID         AS performance_experience_id,
        s.Is_DBOXflag                       AS is_dbox_flag
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc
        ON t.LocationId = loc.LocationID
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d
        ON t.TransactionDateId = d.dateid
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_session s
        ON t.sessionskey = s.sessionskey
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_ticket_type tk
        ON t.TicketTypeSkey = tk.tickettypeskey
    CROSS JOIN month_bounds mb
    WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
      AND t.TicketFLAG = 'Yes'
      AND d.datefull BETWEEN mb.month_start AND mb.month_end
),

-- Format classification (top-tier mapping wins; else 3D; else Regular)
-- DBox is evaluated independently below -- not part of this waterfall
ticket_events_classified AS (
    SELECT
        *,
        CASE
            WHEN auditorium_experience_id = 2      THEN 'AVX'
            WHEN auditorium_experience_id = 3      THEN 'IMAX'
            WHEN auditorium_experience_id = 4      THEN 'VIP'
            WHEN auditorium_experience_id IN (5,6) THEN 'ALT'
            WHEN auditorium_experience_id = 7      THEN 'FourDX'
            WHEN auditorium_experience_id = 8      THEN 'ScreenX'
            WHEN performance_experience_id = 2     THEN 'ThreeD'
            ELSE 'Regular'
        END AS final_format
    FROM ticket_events
),

-- ----------------------------------------------------------------------------
-- Per-population ticket aggregation
-- tickets_free_redeemed / companion_tickets_redeemed / the *_999 / *_1099
-- sub-splits are CC-only concepts -- set NULL here for Scene/Total_Biz, not zero
-- CC and Scene apply CDE_ID IS NOT NULL explicitly; Total_Biz does not.
-- ----------------------------------------------------------------------------
tickets_agg AS (
    SELECT
        'CC' AS population_type,
        COUNT(DISTINCT cde_id)                                                                     AS distinct_members,
        COUNT(DISTINCT cde_id, datefull)                                                           AS total_visits,
        SUM(CASE WHEN final_format = 'Regular' THEN quantity ELSE 0 END)                          AS tickets_regular,
        SUM(CASE WHEN final_format != 'Regular' THEN quantity ELSE 0 END)                          AS tickets_upgrades,
        SUM(CASE WHEN is_dbox_flag = 'Yes' THEN quantity ELSE 0 END)                               AS tickets_dbox,
        SUM(CASE WHEN final_format = 'AVX' THEN quantity ELSE 0 END)                               AS tickets_avx,
        SUM(CASE WHEN final_format = 'IMAX' THEN quantity ELSE 0 END)                              AS tickets_imax,
        SUM(CASE WHEN final_format = 'VIP' THEN quantity ELSE 0 END)                               AS tickets_vip,
        SUM(CASE WHEN final_format = 'ALT' THEN quantity ELSE 0 END)                               AS tickets_alt,
        SUM(CASE WHEN final_format = 'FourDX' THEN quantity ELSE 0 END)                            AS tickets_fourdx,
        SUM(CASE WHEN final_format = 'ScreenX' THEN quantity ELSE 0 END)                           AS tickets_screenx,
        SUM(CASE WHEN final_format = 'ThreeD' THEN quantity ELSE 0 END)                            AS tickets_threed,
        SUM(CASE WHEN paid_flag IS DISTINCT FROM 'Yes' THEN quantity ELSE 0 END)                   AS tickets_free,
        SUM(CASE WHEN paid_flag = 'Yes' THEN quantity ELSE 0 END)                                  AS tickets_paid,
        SUM(CASE WHEN discount_type_id IN (6,8) THEN quantity ELSE 0 END)                          AS tickets_free_redeemed,
        SUM(CASE WHEN discount_type_id IN (5,7) THEN quantity ELSE 0 END)                          AS companion_tickets_redeemed,
        SUM(CASE WHEN discount_type_id = 1 THEN quantity ELSE 0 END)                               AS tickets_undiscounted,
        SUM(CASE WHEN discount_type_id = 6 THEN quantity ELSE 0 END)                               AS tickets_free_999,
        SUM(CASE WHEN discount_type_id = 8 THEN quantity ELSE 0 END)                               AS tickets_free_1099,
        SUM(CASE WHEN discount_type_id = 5 THEN quantity ELSE 0 END)                               AS tickets_companion_999,
        SUM(CASE WHEN discount_type_id = 7 THEN quantity ELSE 0 END)                               AS tickets_companion_1099
    FROM ticket_events_classified
    WHERE cineclub_flag = 'Yes' AND cde_id IS NOT NULL

    UNION ALL

    SELECT
        'Scene' AS population_type,
        COUNT(DISTINCT cde_id),
        COUNT(DISTINCT cde_id, datefull),
        SUM(CASE WHEN final_format = 'Regular' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format != 'Regular' THEN quantity ELSE 0 END),
        SUM(CASE WHEN is_dbox_flag = 'Yes' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'AVX' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'IMAX' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'VIP' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'ALT' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'FourDX' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'ScreenX' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'ThreeD' THEN quantity ELSE 0 END),
        SUM(CASE WHEN paid_flag IS DISTINCT FROM 'Yes' THEN quantity ELSE 0 END),
        SUM(CASE WHEN paid_flag = 'Yes' THEN quantity ELSE 0 END),
        CAST(NULL AS BIGINT),
        CAST(NULL AS BIGINT),
        CAST(NULL AS BIGINT),
        CAST(NULL AS BIGINT),
        CAST(NULL AS BIGINT),
        CAST(NULL AS BIGINT),
        CAST(NULL AS BIGINT)
    FROM ticket_events_classified
    -- anti-join vs CC, same mutual-exclusivity pattern as scene_balance
    WHERE scene_flag = 'Yes' AND cineclub_flag IS DISTINCT FROM 'Yes' AND cde_id IS NOT NULL

    UNION ALL

    SELECT
        'Total_Biz' AS population_type,
        COUNT(DISTINCT cde_id),
        COUNT(DISTINCT cde_id, datefull),
        SUM(CASE WHEN final_format = 'Regular' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format != 'Regular' THEN quantity ELSE 0 END),
        SUM(CASE WHEN is_dbox_flag = 'Yes' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'AVX' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'IMAX' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'VIP' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'ALT' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'FourDX' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'ScreenX' THEN quantity ELSE 0 END),
        SUM(CASE WHEN final_format = 'ThreeD' THEN quantity ELSE 0 END),
        SUM(CASE WHEN paid_flag IS DISTINCT FROM 'Yes' THEN quantity ELSE 0 END),
        SUM(CASE WHEN paid_flag = 'Yes' THEN quantity ELSE 0 END),
        CAST(NULL AS BIGINT),
        CAST(NULL AS BIGINT),
        CAST(NULL AS BIGINT),
        CAST(NULL AS BIGINT),
        CAST(NULL AS BIGINT),
        CAST(NULL AS BIGINT),
        CAST(NULL AS BIGINT)
    FROM ticket_events_classified
    -- No CDE_ID filter: Total_Biz includes anonymous/unidentified ticket buyers
),

-- ----------------------------------------------------------------------------
-- Credited / in-market -- CC-only
-- tickets_credited uses a MEMBER-LEVEL join (current vs prior snapshot per
-- cc_user_profile_id), not a population-level SUM delta. A population delta
-- would net a churned member's entire lifetime recognitions_earned OUT of
-- the current-month sum, understating (or even inverting) the real signal.
-- Summed only over members present in the CURRENT snapshot -- churned
-- members simply drop out rather than dragging the total down.
--
-- Snapshot now carries separate 9.99 (base columns) and 10.99 (_10_99
-- columns) counters. tickets_credited/tickets_in_market = 9.99 + 10.99.
-- COALESCE guards against the 10.99 side being entirely NULL (e.g. feature
-- not yet populated for a given month) so it doesn't null out the total.
-- ----------------------------------------------------------------------------
snapshot_current_members AS (
    SELECT
        cc_user_profile_id,
        recognitions_earned,
        recognitions_earned_10_99,
        recognitions_left,
        recognitions_left_10_99
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
    CROSS JOIN month_bounds mb
    WHERE sn.snapshot_date_id = DATE_FORMAT(mb.month_end, 'yyyyMMdd')
),

snapshot_prior_members AS (
    SELECT
        cc_user_profile_id,
        recognitions_earned        AS recognitions_earned_prior,
        recognitions_earned_10_99  AS recognitions_earned_10_99_prior
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
    CROSS JOIN month_bounds mb
    WHERE sn.snapshot_date_id = DATE_FORMAT(mb.prior_month_end, 'yyyyMMdd')
),

recognitions AS (
    SELECT
        SUM(GREATEST(cur.recognitions_earned - COALESCE(pr.recognitions_earned_prior, 0), 0))                          AS tickets_credited_999,
        SUM(GREATEST(COALESCE(cur.recognitions_earned_10_99, 0) - COALESCE(pr.recognitions_earned_10_99_prior, 0), 0)) AS tickets_credited_1099,
        SUM(cur.recognitions_left)                                                                                     AS tickets_in_market_999,
        SUM(COALESCE(cur.recognitions_left_10_99, 0))                                                                  AS tickets_in_market_1099
    FROM snapshot_current_members cur
    LEFT JOIN snapshot_prior_members pr
        ON cur.cc_user_profile_id = pr.cc_user_profile_id
)

-- ============================================================================
-- Final assembly
-- ============================================================================
SELECT
    mb.month_start AS report_month,
    ta.population_type,
    ta.distinct_members,
    ta.total_visits,
    ta.tickets_regular,
    ta.tickets_upgrades,
    ta.tickets_dbox,
    ta.tickets_avx,
    ta.tickets_imax,
    ta.tickets_vip,
    ta.tickets_alt,
    ta.tickets_fourdx,
    ta.tickets_screenx,
    ta.tickets_threed,
    ta.tickets_free,
    ta.tickets_paid,
    ta.tickets_free_redeemed,
    ta.companion_tickets_redeemed,
    ta.tickets_undiscounted,
    CASE WHEN ta.population_type = 'CC' THEN ta.tickets_free_999      / NULLIF(ta.tickets_free_999 + ta.tickets_free_1099, 0)           ELSE NULL END AS share_9_99_Free,
    CASE WHEN ta.population_type = 'CC' THEN ta.tickets_free_1099     / NULLIF(ta.tickets_free_999 + ta.tickets_free_1099, 0)           ELSE NULL END AS share_10_99_Free,
    CASE WHEN ta.population_type = 'CC' THEN ta.tickets_companion_999 / NULLIF(ta.tickets_companion_999 + ta.tickets_companion_1099, 0) ELSE NULL END AS share_9_99_Companion,
    CASE WHEN ta.population_type = 'CC' THEN ta.tickets_companion_1099/ NULLIF(ta.tickets_companion_999 + ta.tickets_companion_1099, 0) ELSE NULL END AS share_10_99_Companion,
    CASE WHEN ta.population_type = 'CC' THEN r.tickets_credited_999 + COALESCE(r.tickets_credited_1099, 0)   ELSE NULL END AS tickets_credited,
    CASE WHEN ta.population_type = 'CC' THEN r.tickets_in_market_999 + COALESCE(r.tickets_in_market_1099, 0) ELSE NULL END AS tickets_in_market,
    CASE WHEN ta.population_type = 'CC' THEN r.tickets_credited_999   / NULLIF(r.tickets_credited_999 + COALESCE(r.tickets_credited_1099, 0), 0)    ELSE NULL END AS share_9_99_Credited,
    CASE WHEN ta.population_type = 'CC' THEN COALESCE(r.tickets_credited_1099, 0)  / NULLIF(r.tickets_credited_999 + COALESCE(r.tickets_credited_1099, 0), 0)    ELSE NULL END AS share_10_99_Credited,
    CASE WHEN ta.population_type = 'CC' THEN r.tickets_in_market_999  / NULLIF(r.tickets_in_market_999 + COALESCE(r.tickets_in_market_1099, 0), 0)  ELSE NULL END AS share_9_99_InMarket,
    CASE WHEN ta.population_type = 'CC' THEN COALESCE(r.tickets_in_market_1099, 0) / NULLIF(r.tickets_in_market_999 + COALESCE(r.tickets_in_market_1099, 0), 0)  ELSE NULL END AS share_10_99_InMarket
FROM tickets_agg ta
CROSS JOIN month_bounds mb
CROSS JOIN recognitions r
ORDER BY ta.population_type;